# AoC 2024 Day 9 — Disk Fragmenter

**Python — two-pointer compaction over the block list**

Puzzle: <https://adventofcode.com/2024/day/9>

---

> **On puzzle text and inputs.** Advent of Code is Eric Wastl's work, and he asks that puzzle text and per-user inputs not be redistributed. So this notebook carries a summary in my own words plus the *published* example, and pulls the real input at runtime from a local cache that is gitignored. Read the puzzle at the link above.

> **Part 1 only.** Advent of Code reveals Part Two only after a correct Part One submission, and this day is unsolved — so part 2's text does not exist to work from yet. Submitting the answer below unlocks it.

---

## The puzzle

The input is a single line of digits describing a disk, alternating **file length** and **free length**, starting with a file. Files are numbered from 0 in the order they appear, so the file at digit index *i* has ID *i/2*.

```
12345   ->   0..111....22222
```

Compact it by repeatedly taking the **rightmost** file block and moving it into the **leftmost** free block, one block at a time, until no gap remains before the last file block. The checksum is then the sum of `position × file ID` over every occupied block, skipping the free ones.

- **Part 1** — the checksum after compaction.

## The approach

Another plain-Python day, and this time the honest reason is **scale, not impossibility**.

The puzzle *describes* a sequential process: move the rightmost file block into the leftmost gap, repeat. Taken literally that is a loop with a dependency on every iteration. But the description collapses — the final layout is just "file blocks in reverse order, poured into the gaps left to right", which a **two-pointer sweep** computes in one pass: `left` walks forward looking for free, `right` walks back looking for occupied, swap. 94,595 blocks, 118,296 pointer moves, 23,702 swaps, **16 ms**.

And it is genuinely expressible in Spark. Explode the disk map to one row per block, `row_number()` over the free positions ascending and over the occupied blocks descending, join on that rank, and the two windows *are* the compaction. It would work. It would also be:

- two full sorts and a join over 94,595 rows,
- more code than the loop, in a shape that needs a paragraph of explanation, and
- slower, because a trivial Spark Connect query costs tens of milliseconds before it reads a single row, and the entire Python solution — parse, compact, checksum — finishes in about 20 ms.

The crossover would come somewhere north of a few million blocks. This input is not there, and pretending otherwise would be the actual mistake.

## Setup

Connect to the cluster's Spark Connect endpoint and import the solution.

In [ ]:
import sys

sys.path.insert(0, '..')  # so `aoc_spark` resolves when running from notebooks/

from aoc_spark.session import get_spark
from aoc_spark.inputs import get_input
from aoc_spark.y2024 import day09

spark = get_spark('aoc-2024-day09')
print('Spark', spark.version)

## The published example

The same data the test suite asserts on.

In [ ]:
EXAMPLE = '2333133121414131402\n'

print('part 1:', day09.part1(spark, EXAMPLE), '(expected 1928)')

### Watching the pointers close

The cell below prints the disk after every single swap on the published example, alongside where the two pointers sit. Compare the last line with the puzzle's final layout — and note that neither pointer ever revisits a position.

In [ ]:
def render(disk):
    return ''.join('.' if b == day09.FREE else str(b) for b in disk)


print('12345    ->', render(day09.blocks('12345')))
print('90909    ->', render(day09.blocks('90909')))
print('EXAMPLE  ->', render(day09.blocks(EXAMPLE)))

disk = day09.blocks(EXAMPLE)
files = sum(1 for b in disk if b != day09.FREE)
print(f'\n{len(disk)} blocks: {files} used, {len(disk) - files} free')

# The two pointers, and the disk after each swap. Every line depends
# on the line above it -- the gap you fill next is the one the previous
# swap did not reach.
left, right = 0, len(disk) - 1
swaps = 0
print()
while left < right:
    if disk[left] != day09.FREE:
        left += 1
    elif disk[right] == day09.FREE:
        right -= 1
    else:
        disk[left], disk[right] = disk[right], day09.FREE
        swaps += 1
        print(f'swap {swaps:2d}  left={left:2d} right={right:2d}  {render(disk)}')

checksum = sum(pos * fid for pos, fid in enumerate(disk) if fid != day09.FREE)
print(f'\n{swaps} swaps, pointers met at {left}, checksum {checksum}')

## The real input

`get_input` is cache-first: local gitignored file → Postgres → adventofcode.com. In practice it hits the local file and never touches the network.

In [ ]:
import time

data = get_input(2024, 9)
print(f'input: {len(data):,} chars, {len(data.splitlines()):,} lines')

started = time.perf_counter()
answer = day09.part1(spark, data)
print(f'part 1: {answer}  ({(time.perf_counter() - started) * 1000:.0f} ms)')

## Cross-check

Days 6+ have no known-good answer, so correctness rests on an independently written plain-Python implementation agreeing with the Spark one. That is evidence, not proof — a shared misreading of the puzzle would survive both.

In [ ]:
from reference_python.y2024 import day09 as reference

cross = reference.part1(data)
print('reference:', cross)
print('agree:    ', cross == answer)

## Notes & gotchas

- The disk map is **positional**: even digit index = a file, odd = a gap. That is the only reason the file ID is `i // 2`. Miscount the alternation by one and every ID shifts.
- **Zero-length runs are legal** and common — a `0` digit in a gap slot means two files touch. `blocks()` handles it by extending with an empty list; anything that assumes every run has length ≥ 1 breaks here.
- `FREE` is `-1`, and the checks are `!= FREE` / `== FREE`, never truthiness. File ID **0 is falsy**, so `if fid:` would silently treat the first file as empty space.
- 19,999 digits expand to 94,595 block entries. The expansion is the memory cost of this approach; the run-based cross-check in `reference_python/y2024/day09.py` avoids it entirely by scoring each placed run with an arithmetic series.
- The input is **one very long line** — `strip()` removes the trailing newline, and `splitlines()` is never called. Any stray whitespace inside would blow up `int()`.
- The checksum is 6,398,252,054,886 — comfortably past 2^32. Python does not care; a Spark port would need `bigint` throughout.
- The loop terminates when the pointers meet. The final `sum` skips `FREE` anyway, so the exact meeting cell needs no special handling.